# Convolutional Neural Network

In [ ]:
import os

def delete_jpg_files():
  """Deletes all .jpg files in the current directory."""
  for file in os.listdir():
    if file.endswith(".jpg"):
      os.remove(file)
      print(f"Deleted {file}")

# You can call the function like this:
# delete_jpg_files()

In [ ]:
from google.colab import files

uploaded = files.upload()

for fn in uploaded.keys():
  print('User uploaded file "{name}" with length {length} bytes'.format(
      name=fn, length=len(uploaded[fn])))

Saving sachet.jpg to sachet.jpg
User uploaded file "sachet.jpg" with length 22068 bytes


In [ ]:
import numpy as np
import cv2

def processImage(image):
    image = cv2.imread(image)
    image = cv2.cvtColor(src=image, code=cv2.COLOR_BGR2GRAY)
    return image

# https://medium.com/analytics-vidhya/2d-convolution-using-python-numpy-43442ff5f381
def convolve2D(image, kernel, padding=0, strides=1):
    # Cross Correlation
    kernel = np.flipud(np.fliplr(kernel))

    # Gather Shapes of Kernel + Image + Padding
    xKernShape = kernel.shape[0]
    yKernShape = kernel.shape[1]
    xImgShape = image.shape[0]
    yImgShape = image.shape[1]

    # Shape of Output Convolution
    xOutput = int(((xImgShape - xKernShape + 2 * padding) / strides) + 1)
    yOutput = int(((yImgShape - yKernShape + 2 * padding) / strides) + 1)
    output = np.zeros((xOutput, yOutput))

    # Apply Equal Padding to All Sides
    if padding != 0:
        imagePadded = np.zeros((image.shape[0] + padding*2, image.shape[1] + padding*2))
        imagePadded[int(padding):int(-1 * padding), int(padding):int(-1 * padding)] = image
        print(imagePadded)
    else:
        imagePadded = image

    # Iterate through image
    for y in range(image.shape[1]):
        # Exit Convolution
        if y > image.shape[1] - yKernShape:
            break
        # Only Convolve if y has gone down by the specified Strides
        if y % strides == 0:
            for x in range(image.shape[0]):
                # Go to next row once kernel is out of bounds
                if x > image.shape[0] - xKernShape:
                    break
                try:
                    # Only Convolve if x has moved by the specified Strides
                    if x % strides == 0:
                        output[x, y] = (kernel * imagePadded[x: x + xKernShape, y: y + yKernShape]).sum()
                except:
                    break

    return output

In [ ]:
image = processImage('sachet.jpg')

In [ ]:
image.shape

(800, 800)

In [ ]:
filtre = np.array([
    [1, 2, 1],
    [0, 0, 0],
    [-1, -2, -1]
])

In [ ]:
output = convolve2D(image, filtre, padding=2)

[[  0.   0.   0. ...   0.   0.   0.]
 [  0.   0.   0. ...   0.   0.   0.]
 [  0.   0. 255. ... 255.   0.   0.]
 ...
 [  0.   0. 255. ... 255.   0.   0.]
 [  0.   0.   0. ...   0.   0.   0.]
 [  0.   0.   0. ...   0.   0.   0.]]


In [ ]:
cv2.imwrite('salon_horizontal.png', output)

True

In [ ]:
filtre = np.array([[-1, 0, 1],
                   [-2, 0, 2],
                   [-1, 0, 1]])

In [ ]:
output = convolve2D(image, filtre, padding=2)
cv2.imwrite('salon_vertical.png', output)


[[  0.   0.   0. ...   0.   0.   0.]
 [  0.   0.   0. ...   0.   0.   0.]
 [  0.   0. 255. ... 255.   0.   0.]
 ...
 [  0.   0. 255. ... 255.   0.   0.]
 [  0.   0.   0. ...   0.   0.   0.]
 [  0.   0.   0. ...   0.   0.   0.]]


True

Test CNN sur les données de Fashion Mnist

In [ ]:
import tensorflow as tf
data = tf.keras.datasets.fashion_mnist
(training_images, training_labels), (test_images, test_labels) = data.load_data()

In [ ]:
training_images = training_images / 255.0
test_images = test_images / 255.0

In [ ]:
training_labels = tf.keras.utils.to_categorical(training_labels)
test_labels = tf.keras.utils.to_categorical(test_labels)

In [ ]:
training_images.shape

In [ ]:
training_images = training_images.reshape((60000,28,28,1))


In [ ]:
training_images.shape

In [ ]:
test_images = test_images.reshape((10000,28,28,1))

In [ ]:
test_images.shape

## Modeling

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import SGD

In [ ]:
model = Sequential([

    #Extraction de caractéristiques
    tf.keras.layers.Conv2D(filters=64, kernel_size=(3, 3), padding="same", activation='relu', input_shape=(28, 28,1)),
    tf.keras.layers.MaxPooling2D(2,2),
    tf.keras.layers.Conv2D(filters=64, kernel_size=(3, 3), activation="relu"),
    tf.keras.layers.MaxPooling2D(2,2),

    # applatir
    tf.keras.layers.Flatten(),

    # Dense
    tf.keras.layers.Dense(units=128, activation="relu"),
    tf.keras.layers.Dense(units=10, activation="softmax"),
                ])

In [ ]:
model.summary()

In [ ]:
6*6*64

In [ ]:
10*64

In [ ]:
64*(64*9) +64

In [ ]:
tf.keras.utils.plot_model(model)

In [ ]:
model_ckp = tf.keras.callbacks.ModelCheckpoint(filepath="best_model.h5",
                            monitor="val_accuracy",
                            mode="max",
                            save_best_only=True)
stop = tf.keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=3)

In [ ]:
model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=['accuracy'])

In [ ]:
h= model.fit(training_images, training_labels, epochs=50,
             validation_data=(test_images, test_labels),
             callbacks = [model_ckp, stop])

In [ ]:
 MSE : loss: 0.0704 - accuracy: 0.5393 - val_loss: 0.0682 - val_accuracy: 0.5872
 CROSS ENTROPY : loss: 0.5450 - accuracy: 0.8112 - val_loss: 0.5563 - val_accuracy: 0.8020
 Relu : loss: 0.4099 - accuracy: 0.8585 - val_loss: 0.4308 - val_accuracy: 0.8479
 (20 epochs) : loss: 0.3009 - accuracy: 0.8936 - val_loss: 0.3549 - val_accuracy: 0.8744
 model complexe : loss: 0.2473 - accuracy: 0.9109 - val_loss: 0.3229 - val_accuracy: 0.8847
 Dropout : loss: 0.3238 - accuracy: 0.8834 - val_loss: 0.3361 - val_accuracy: 0.8800
 Adam : loss: 0.8879 - accuracy: 0.6822 - val_loss: 0.6868 - val_accuracy: 0.7686
 Callbacks : loss: 0.3065 - accuracy: 0.8892 - val_loss: 0.3300 - val_accuracy: 0.8801

 CNN : loss: 0.1843 - accuracy: 0.9309 - val_loss: 0.2286 - val_accuracy: 0.9194